<a href="https://colab.research.google.com/github/maierav/claupenscope/blob/main/notebooks/rf_population_allprobes_001637.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Cross-session population RF analysis — all probes, no letter assumption

**Motivation.** Probe letters (A, B, …) are per-session device IDs assigned at insertion. They do **not** map to a fixed anatomical target across animals. A summary that pools "all Probe-A units across subjects" therefore quietly assumes a correspondence the data does not guarantee. This notebook drops that assumption and analyses the population at the **insertion** level (one row per `session × probe`) and at the **unit** level pooled across all probes — coloured by subject and labelled by anatomical CCF coordinates when present.

**Data.** OpenScope Predictive Processing dandiset 001637, all `rf_mapping` blocks across all sessions and all probes (4 to 6 per session).

**Cache.** `results/rf_cache_allprobes/<asset_id[:8]>.pkl` — per-unit summary plus session metadata. Resume-safe: re-runs only stream NWBs missing from cache.

**Figures.**
1. **Insertion RF centroids** in azimuth/elevation, one dot per `(session × probe)`, coloured by subject. Reveals whether probe insertions consistently target similar retinotopic regions across animals.
2. **Per-subject probe spread** — small multiples; each panel shows one subject's insertions. Tests within-subject targeting consistency.
3. **Pooled retinotopic coverage** — 2-D density of all visual-unit RF centres, no probe-letter colouring.
4. **Retinotopy in CCF space** (if `estimated_x/y/z` are present) — top-down ML/AP scatter coloured by preferred azimuth. Physical-space test of retinotopic organisation.
5. **Insertion-level quality summary** — per-subject box plots of n SUA, n visual, median peak ΔFR, retinotopic spread. Insertion is the unit of observation.
6. **Visual fraction vs depth per session** — one line per session, coloured by subject, all probes pooled within each session.

All outputs as vector PDF + raster PNG.

In [ ]:
!pip install -q git+https://github.com/maierav/claupenscope.git \
    dandi remfile h5py pynwb scipy matplotlib numpy pandas

In [ ]:
import os, sys, time, pickle, gc, glob
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.gridspec import GridSpec
from scipy.ndimage import gaussian_filter
from dandi.dandiapi import DandiAPIClient

from openscope_pp.loaders.streaming import open_nwb
from openscope_pp.loaders.trials import load_trials

mpl.rcParams.update({
    "figure.dpi": 130, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.family": "serif",
    "font.serif": ["DejaVu Serif", "Liberation Serif", "serif"],
    "mathtext.fontset": "dejavuserif",
    "font.size": 9.5, "axes.labelsize": 10, "axes.titlesize": 10.5,
    "axes.linewidth": 0.8, "axes.spines.top": False, "axes.spines.right": False,
    "xtick.labelsize": 8.5, "ytick.labelsize": 8.5,
    "xtick.major.size": 3, "ytick.major.size": 3,
    "xtick.major.width": 0.8, "ytick.major.width": 0.8,
    "legend.fontsize": 8.0, "legend.frameon": False,
    "pdf.fonttype": 42, "ps.fonttype": 42, "svg.fonttype": "none",
})

RF_WIN     = (-0.30, 0.50)
BL_WIN     = (-0.20, 0.00)
RESP_WIN   = (0.03, 0.20)
BIN_SIZE   = 0.010
RF_SMOOTH  = 0.8
VIS_THR    = 5.0   # peak ΔFR threshold for "visual"

DANDISET_ID = "001637"
CACHE_DIR   = "results/rf_cache_allprobes"
os.makedirs(CACHE_DIR, exist_ok=True)

# Limit how many uncached sessions to process this run; None = all
N_NEW_THIS_RUN = None

print("Setup complete. Cache:", CACHE_DIR)

## 1. Stream + cache per-session RF summaries (all probes)

Per cached file we store a per-unit table including each unit's full `device_name`, depth, peak ΔFR, RF centre, half-max area, and CCF coordinates `estimated_x/y/z` when the units table exposes them. *No probe-letter filter.*

In [ ]:
def decode_str(arr):
    if arr.dtype.kind in ("S", "O"):
        return np.array([v.decode() if isinstance(v, bytes) else str(v) for v in arr])
    return arr

def bin_spikes(uid_arr, spikes, index, onsets, window, bin_size):
    pre, post = window
    edges   = np.arange(pre, post + bin_size, bin_size)
    centers = 0.5 * (edges[:-1] + edges[1:])
    out = np.zeros((len(onsets), len(uid_arr), len(centers)), dtype=np.float32)
    for j, uid in enumerate(uid_arr):
        i0 = int(index[uid - 1]) if uid > 0 else 0
        s  = spikes[i0:int(index[uid])]
        if not len(s): continue
        for i, t0 in enumerate(onsets):
            lo = int(np.searchsorted(s, t0 + pre))
            hi = int(np.searchsorted(s, t0 + post))
            if lo < hi:
                cnt, _ = np.histogram(s[lo:hi] - t0, bins=edges)
                out[i, j, :] = cnt / bin_size
    return out, centers

def _safe_col(units_grp, name):
    "Return the units column as float64 ndarray, or all-NaN if missing."
    if name in units_grp:
        return np.asarray(units_grp[name][:]).astype(float)
    return None

def process_session(asset, force=False):
    asset_id   = asset.identifier
    subject_id = asset.path.split("/")[0]
    cache_path = os.path.join(CACHE_DIR, f"{asset_id[:8]}.pkl")
    if os.path.exists(cache_path) and not force:
        return cache_path, "cached"

    t0 = time.time()
    handle = open_nwb(asset_id)
    try:
        trials = load_trials(handle)
        h5     = handle.h5
        rf_trials = trials[trials["block_kind"] == "rf_mapping"].copy()
        if len(rf_trials) < 50:
            return cache_path, f"skip (rf trials={len(rf_trials)})"

        xs_grid = np.sort(rf_trials["x"].dropna().unique())
        ys_grid = np.sort(rf_trials["y"].dropna().unique())

        u = h5["units"]
        decoder_label = decode_str(u["decoder_label"][:])
        default_qc    = u["default_qc"][:].astype(bool)
        device_name   = decode_str(u["device_name"][:])
        spike_times   = u["spike_times"][:]
        spk_index     = u["spike_times_index"][:]
        depth_um      = _safe_col(u, "depth")
        ccf_x         = _safe_col(u, "estimated_x")
        ccf_y         = _safe_col(u, "estimated_y")
        ccf_z         = _safe_col(u, "estimated_z")

        if depth_um is None:
            depth_um = np.full(len(decoder_label), np.nan)
        for arr_name, arr in (("ccf_x", ccf_x), ("ccf_y", ccf_y), ("ccf_z", ccf_z)):
            if arr is None:
                locals()[arr_name]  # no-op; we just create NaN below
        ccf_x = ccf_x if ccf_x is not None else np.full(len(decoder_label), np.nan)
        ccf_y = ccf_y if ccf_y is not None else np.full(len(decoder_label), np.nan)
        ccf_z = ccf_z if ccf_z is not None else np.full(len(decoder_label), np.nan)

        is_sua  = (decoder_label == "sua") & default_qc
        sua_idx = np.where(is_sua)[0]
        if not len(sua_idx):
            return cache_path, "skip (no SUA)"

        x_trial = rf_trials["x"].values
        y_trial = rf_trials["y"].values
        onsets  = rf_trials["start_time"].values
        arr, _t = bin_spikes(sua_idx, spike_times, spk_index, onsets, RF_WIN, BIN_SIZE)

        rsp = (_t >= RESP_WIN[0]) & (_t < RESP_WIN[1])
        bl  = (_t >= BL_WIN[0])   & (_t < BL_WIN[1])

        n_units = len(sua_idx)
        rf_raw  = np.zeros((n_units, len(ys_grid), len(xs_grid)), dtype=np.float32)
        for xi, x in enumerate(xs_grid):
            for yi, y in enumerate(ys_grid):
                m = (x_trial == x) & (y_trial == y)
                if not m.any(): continue
                rf_raw[:, yi, xi] = (np.nanmean(arr[m][:, :, rsp], axis=(0, 2))
                                    - np.nanmean(arr[m][:, :, bl ], axis=(0, 2)))
        del arr; gc.collect()

        rf_sm   = np.array([gaussian_filter(rf_raw[k], RF_SMOOTH) for k in range(n_units)])
        peak    = rf_sm.reshape(n_units, -1).max(axis=1)
        flat_am = rf_sm.reshape(n_units, -1).argmax(axis=1)
        yi      = flat_am // len(xs_grid)
        xi      = flat_am %  len(xs_grid)
        cx, cy  = xs_grid[xi], ys_grid[yi]
        cell_area = (float((xs_grid[1] - xs_grid[0]) * (ys_grid[1] - ys_grid[0]))
                     if len(xs_grid) > 1 and len(ys_grid) > 1 else float("nan"))
        hm_area = np.array([
            (rf_sm[k] >= 0.5 * peak[k]).sum() * cell_area if peak[k] > 0 else np.nan
            for k in range(n_units)
        ], dtype=float)

        df = pd.DataFrame({
            "unit_idx":     sua_idx,
            "device_name":  device_name[sua_idx],
            "depth_um":     depth_um[sua_idx],
            "ccf_x":        ccf_x[sua_idx],
            "ccf_y":        ccf_y[sua_idx],
            "ccf_z":        ccf_z[sua_idx],
            "peak_dfr":     peak,
            "rf_x":         cx,
            "rf_y":         cy,
            "rf_hm_area":   hm_area,
        })
        meta = {
            "asset_id":    asset_id,
            "subject_id":  subject_id,
            "asset_path":  asset.path,
            "xs_grid":     xs_grid,
            "ys_grid":     ys_grid,
            "n_rf_trials": int(len(rf_trials)),
            "runtime_s":   time.time() - t0,
            "probes_in_session": sorted(np.unique(device_name).tolist()),
        }
        with open(cache_path, "wb") as fh:
            pickle.dump({"meta": meta, "units": df}, fh)
        return cache_path, f"new ({len(df)} units, {time.time()-t0:.0f} s)"
    finally:
        try: handle.close()
        except Exception: pass

client = DandiAPIClient()
ds     = client.get_dandiset(DANDISET_ID)
assets = sorted(list(ds.get_assets()), key=lambda a: a.path)
print(f"Dandiset {DANDISET_ID}: {len(assets)} assets")

n_new = 0
for ai, a in enumerate(assets):
    cache_path, status = process_session(a)
    print(f"  [{ai+1:>2}/{len(assets)}] {a.path.split('/')[-1][:55]:55s}  {status}")
    if status.startswith("new"):
        n_new += 1
        if N_NEW_THIS_RUN is not None and n_new >= N_NEW_THIS_RUN:
            print(f"  reached N_NEW_THIS_RUN={N_NEW_THIS_RUN}; stopping early")
            break
print(f"\nProcessed {n_new} new sessions this run.")

## 2. Aggregate cache → unit table + insertion table

We build two DataFrames: `UNITS` (one row per SUA) and `INSERT` (one row per *insertion* = `subject × asset_id × device_name`). Insertion-level metrics — yield, RF centroid, retinotopic spread, median depth — are the right unit of observation when probe letters do not pool across animals.

In [ ]:
rows = []
session_meta = []
for fp in sorted(glob.glob(os.path.join(CACHE_DIR, "*.pkl"))):
    with open(fp, "rb") as fh: d = pickle.load(fh)
    df = d["units"].copy()
    md = d["meta"]
    df["asset_id"]   = md["asset_id"][:8]
    df["subject_id"] = md["subject_id"]
    rows.append(df); session_meta.append(md)
if not rows:
    raise RuntimeError("No cache files — run the previous cell first.")
UNITS = pd.concat(rows, ignore_index=True)
UNITS["is_visual"] = UNITS["peak_dfr"] >= VIS_THR

# Insertion = subject × session × probe (device_name as-is, no letter assumption)
def _circ_or_med(values):
    v = np.asarray(values, float); v = v[np.isfinite(v)]
    return float(np.median(v)) if len(v) else float("nan")

INSERT = (UNITS.groupby(["subject_id", "asset_id", "device_name"], as_index=False)
                .agg(n_sua=("unit_idx", "size"),
                     n_visual=("is_visual", "sum"),
                     median_depth=("depth_um", "median"),
                     median_peak=("peak_dfr", "median")))
INSERT["visual_frac"] = INSERT["n_visual"] / INSERT["n_sua"].clip(lower=1)

# RF centroid + spread per insertion (visual units only)
vis = UNITS[UNITS["is_visual"]]
centroid = (vis.groupby(["subject_id", "asset_id", "device_name"])
                .agg(rf_x_med=("rf_x", "median"),
                     rf_y_med=("rf_y", "median"),
                     rf_x_sd =("rf_x", "std"),
                     rf_y_sd =("rf_y", "std"),
                     rf_hm_med=("rf_hm_area", "median"))).reset_index()
centroid["rf_spread"] = np.sqrt(centroid["rf_x_sd"].fillna(0)**2 +
                                  centroid["rf_y_sd"].fillna(0)**2)
INSERT = INSERT.merge(centroid, on=["subject_id", "asset_id", "device_name"], how="left")

n_subj = UNITS["subject_id"].nunique()
n_sess = UNITS["asset_id"].nunique()
print(f"  {n_subj} subjects · {n_sess} sessions · {len(UNITS)} SUA · {int(UNITS['is_visual'].sum())} visual")
print(f"  insertions = subject × session × probe : {len(INSERT)} rows")
print("\nProbes per session (showing first 8 sessions):")
for md in session_meta[:8]:
    print(f"  {md['asset_id'][:8]} {md['subject_id']:14s}: {md['probes_in_session']}")

# Stable colour per subject (categorical palette of length up to 14)
subjects = sorted(UNITS["subject_id"].unique())
_palette = (mpl.colormaps["tab20"].colors + mpl.colormaps["tab20b"].colors)
SUBJECT_COLOR = {s: _palette[i % len(_palette)] for i, s in enumerate(subjects)}
print(f"  {len(subjects)} subjects, palette assigned")

## Figure 1 — Insertion RF centroids in azimuth/elevation

One dot per insertion (`session × probe`). Position = median RF of that probe's visual units. Colour = subject. Marker shape = the *trailing* letter of the probe device name, only as a within-session disambiguator — *no* identity is assumed across subjects. Marginal histograms summarise the population per axis.

In [ ]:
# Use the most common stimulus grid for axis padding
grids = [(tuple(m["xs_grid"]), tuple(m["ys_grid"])) for m in session_meta]
common_grid = max(set(grids), key=grids.count) if grids else (None, None)
if common_grid[0]:
    xs_g = np.asarray(common_grid[0]); ys_g = np.asarray(common_grid[1])
    xlim = (xs_g[0] - 5, xs_g[-1] + 5); ylim = (ys_g[0] - 5, ys_g[-1] + 5)
else:
    xlim = (-50, 50); ylim = (-50, 50)

_marker_for = lambda dn: {"A": "o", "B": "s", "C": "^", "D": "D",
                          "E": "P", "F": "X"}.get(dn.strip()[-1].upper(), "o")

fig = plt.figure(figsize=(7.0, 6.0))
gs  = GridSpec(2, 2, width_ratios=[5, 1], height_ratios=[1, 5],
               hspace=0.03, wspace=0.03, figure=fig)
ax  = fig.add_subplot(gs[1, 0])
ax_top = fig.add_subplot(gs[0, 0], sharex=ax)
ax_rgt = fig.add_subplot(gs[1, 1], sharey=ax)

for _, row in INSERT.dropna(subset=["rf_x_med", "rf_y_med"]).iterrows():
    ax.scatter(row["rf_x_med"], row["rf_y_med"],
               s=42, color=SUBJECT_COLOR[row["subject_id"]],
               marker=_marker_for(row["device_name"]),
               edgecolor="white", linewidth=0.6, alpha=0.9, rasterized=True)

ax.axhline(0, color="0.6", lw=0.5, ls=":")
ax.axvline(0, color="0.6", lw=0.5, ls=":")
ax.set_xlabel("Median RF azimuth (°)")
ax.set_ylabel("Median RF elevation (°)")
ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect("equal", adjustable="box")

if common_grid[0]:
    x_edges = np.append(xs_g - (xs_g[1]-xs_g[0])/2, xs_g[-1] + (xs_g[1]-xs_g[0])/2)
    y_edges = np.append(ys_g - (ys_g[1]-ys_g[0])/2, ys_g[-1] + (ys_g[1]-ys_g[0])/2)
else:
    x_edges = np.linspace(*xlim, 14); y_edges = np.linspace(*ylim, 14)
ax_top.hist(INSERT["rf_x_med"].dropna(), bins=x_edges, color="0.4",
            histtype="stepfilled", alpha=0.8, edgecolor="0.2", lw=0.6)
ax_rgt.hist(INSERT["rf_y_med"].dropna(), bins=y_edges, color="0.4",
            histtype="stepfilled", alpha=0.8, edgecolor="0.2", lw=0.6,
            orientation="horizontal")
for ax2 in (ax_top, ax_rgt):
    ax2.tick_params(labelleft=False, labelbottom=False, length=0)
    for s in ax2.spines.values(): s.set_visible(False)
ax_top.set_yticks([]); ax_rgt.set_xticks([])

# Subject + marker legend (compact, outside the main scatter)
from matplotlib.lines import Line2D
subj_handles = [Line2D([0],[0], marker="o", color="w", markerfacecolor=c,
                       markeredgecolor="white", markeredgewidth=0.5, ms=7,
                       label=s.replace("sub-",""))
                for s, c in SUBJECT_COLOR.items()]
marker_handles = [Line2D([0],[0], marker=_marker_for("X"+ltr), color="k",
                          ms=7, lw=0, label=f"…{ltr}")
                  for ltr in "ABCDEF"]
leg1 = fig.legend(handles=subj_handles, title="Subject", loc="upper left",
                  bbox_to_anchor=(0.96, 0.95), fontsize=7.5, title_fontsize=8.5)
fig.add_artist(leg1)
fig.legend(handles=marker_handles, title="Probe (last char)",
           loc="upper left", bbox_to_anchor=(0.96, 0.45),
           fontsize=7.5, title_fontsize=8.5)

fig.suptitle("Figure 1 | Insertion RF centroids — colour = subject, marker = device-name suffix",
             y=0.995, fontweight="bold")
fig.savefig("rfpop_all_fig1_insertion_centroids.pdf", dpi=200)
fig.savefig("rfpop_all_fig1_insertion_centroids.png")
plt.show(); plt.close(fig); gc.collect()

## Figure 2 — Per-subject insertion spread

Small multiples, one panel per subject. Each panel shows that subject's insertions in az/el (one dot per `session × probe`). Tight clusters → consistent targeting across days; spread → either deliberate retinotopic coverage or insertion variability.

In [ ]:
n_subj = len(subjects)
ncols = min(5, n_subj)
nrows = int(np.ceil(n_subj / ncols))
fig, axes = plt.subplots(nrows, ncols, figsize=(ncols*2.4 + 0.6, nrows*2.4 + 0.6),
                          sharex=True, sharey=True, squeeze=False)
for ax, s in zip(axes.flat, subjects):
    sub = INSERT[INSERT["subject_id"] == s].dropna(subset=["rf_x_med", "rf_y_med"])
    c = SUBJECT_COLOR[s]
    ax.axhline(0, color="0.7", lw=0.4, ls=":")
    ax.axvline(0, color="0.7", lw=0.4, ls=":")
    if len(sub):
        for _, row in sub.iterrows():
            ax.scatter(row["rf_x_med"], row["rf_y_med"],
                       s=30, color=c, edgecolor="white", linewidth=0.5,
                       marker=_marker_for(row["device_name"]),
                       alpha=0.92, rasterized=True)
    ax.set_xlim(xlim); ax.set_ylim(ylim); ax.set_aspect("equal", adjustable="box")
    ax.set_title(s.replace("sub-", ""), fontsize=9, color=c, fontweight="bold")
    ax.tick_params(labelsize=7)
for ax in axes.flat[n_subj:]: ax.axis("off")
for r in range(nrows):
    axes[r, 0].set_ylabel("El (°)")
for c in range(ncols):
    axes[-1, c].set_xlabel("Az (°)")
fig.suptitle("Figure 2 | Per-subject probe insertions in retinotopic space",
             y=1.0, fontweight="bold")
fig.savefig("rfpop_all_fig2_per_subject_spread.pdf", dpi=200)
fig.savefig("rfpop_all_fig2_per_subject_spread.png")
plt.show(); plt.close(fig); gc.collect()

## Figure 3 — Pooled retinotopic coverage (no probe-letter pooling)

Density of *all visual-unit RF centres* across every insertion. This is the question "what part of the visual field did the dataset cover, period?" — answered without committing to any probe-letter identity.

In [ ]:
vis = UNITS[UNITS["is_visual"]].dropna(subset=["rf_x", "rf_y"])
if not len(vis):
    print("No visual units — skipping Figure 3.")
else:
    fig, ax = plt.subplots(figsize=(5.5, 4.6))
    if common_grid[0]:
        xb = np.append(xs_g - (xs_g[1]-xs_g[0])/2, xs_g[-1] + (xs_g[1]-xs_g[0])/2)
        yb = np.append(ys_g - (ys_g[1]-ys_g[0])/2, ys_g[-1] + (ys_g[1]-ys_g[0])/2)
    else:
        xb = np.linspace(*xlim, 12); yb = np.linspace(*ylim, 12)
    H, _xe, _ye = np.histogram2d(vis["rf_x"].values, vis["rf_y"].values, bins=[xb, yb])
    im = ax.pcolormesh(_xe, _ye, H.T, cmap="viridis", shading="auto")
    ax.axhline(0, color="white", lw=0.5, ls=":")
    ax.axvline(0, color="white", lw=0.5, ls=":")
    ax.set_xlabel("Azimuth (°)"); ax.set_ylabel("Elevation (°)")
    ax.set_aspect("equal", adjustable="box")
    cb = fig.colorbar(im, ax=ax, fraction=0.045, pad=0.03)
    cb.set_label("Visual units (count)")
    cb.outline.set_linewidth(0.4)
    fig.suptitle(f"Figure 3 | Pooled RF coverage  ·  {len(vis):,} visual units across all insertions",
                 y=0.99, fontweight="bold")
    fig.savefig("rfpop_all_fig3_pooled_coverage.pdf", dpi=200)
    fig.savefig("rfpop_all_fig3_pooled_coverage.png")
    plt.show(); plt.close(fig); gc.collect()

## Figure 4 — Retinotopy in CCF space (if available)

If the units table exposes `estimated_x/y/z` (Allen CCF), we project all visual units to a top-down ML/AP scatter (using `ccf_z` for ML and `ccf_x` for AP — Allen-convention) and colour each unit by its preferred azimuth. A clean colour gradient indicates retinotopic organisation in *physical* space — the right cross-session test, completely independent of probe letters.

In [ ]:
vis = UNITS[UNITS["is_visual"]]
have_ccf = vis[["ccf_x", "ccf_y", "ccf_z"]].notna().all(axis=1)
vis_ccf = vis[have_ccf]
if not len(vis_ccf):
    print("No visual units have valid CCF coordinates — skipping Figure 4.")
else:
    fig, axes = plt.subplots(1, 2, figsize=(9.5, 4.4))
    sc1 = axes[0].scatter(vis_ccf["ccf_z"], vis_ccf["ccf_x"],
                          c=vis_ccf["rf_x"], cmap="twilight",
                          s=12, alpha=0.7, edgecolor="none", rasterized=True)
    axes[0].invert_yaxis()  # AP axis convention (anterior up)
    axes[0].set_xlabel("CCF medio-lateral z (µm)")
    axes[0].set_ylabel("CCF anterior-posterior x (µm)")
    axes[0].set_aspect("equal", adjustable="box")
    axes[0].set_title("top-down (ML × AP), colour = RF azimuth", loc="left", fontsize=9.5)
    cb1 = fig.colorbar(sc1, ax=axes[0], fraction=0.045, pad=0.02)
    cb1.set_label("Preferred azimuth (°)")
    cb1.outline.set_linewidth(0.4)

    sc2 = axes[1].scatter(vis_ccf["ccf_z"], vis_ccf["ccf_x"],
                          c=vis_ccf["rf_y"], cmap="twilight_shifted",
                          s=12, alpha=0.7, edgecolor="none", rasterized=True)
    axes[1].invert_yaxis()
    axes[1].set_xlabel("CCF medio-lateral z (µm)")
    axes[1].set_ylabel("CCF anterior-posterior x (µm)")
    axes[1].set_aspect("equal", adjustable="box")
    axes[1].set_title("top-down (ML × AP), colour = RF elevation", loc="left", fontsize=9.5)
    cb2 = fig.colorbar(sc2, ax=axes[1], fraction=0.045, pad=0.02)
    cb2.set_label("Preferred elevation (°)")
    cb2.outline.set_linewidth(0.4)

    fig.suptitle(f"Figure 4 | Retinotopic organisation in CCF space  ·  {len(vis_ccf):,} visual units",
                 y=1.0, fontweight="bold")
    fig.savefig("rfpop_all_fig4_ccf_retinotopy.pdf", dpi=200)
    fig.savefig("rfpop_all_fig4_ccf_retinotopy.png")
    plt.show(); plt.close(fig); gc.collect()

## Figure 5 — Insertion-level quality summary

Each insertion contributes one observation. Box plots per subject of (a) n SUA, (b) n visual, (c) median peak ΔFR, (d) retinotopic spread (deg). Subjects with consistently low retinotopic spread placed all their probes in nearby retinotopic regions; subjects with high spread either targeted multiple regions or had variable insertions.

In [ ]:
metrics = [
    ("n_sua",       "Units per insertion",      False),
    ("n_visual",    "Visual units per insertion", False),
    ("median_peak", "Median peak ΔFR (spk·s$^{-1}$)", True),
    ("rf_spread",   "Retinotopic spread (°)",   False),
]
fig, axes = plt.subplots(2, 2, figsize=(11, 7), sharey=False)
for ax, (col, label, log_y) in zip(axes.flat, metrics):
    data = []
    keep_subjects = []
    for s in subjects:
        v = INSERT.loc[INSERT["subject_id"] == s, col].values
        v = v[np.isfinite(v.astype(float))]
        if len(v) >= 1:
            data.append(v.astype(float)); keep_subjects.append(s)
    pos = np.arange(len(keep_subjects))
    parts = ax.boxplot(data, positions=pos, widths=0.65,
                       patch_artist=True, showcaps=False,
                       medianprops=dict(color="black", lw=1.2),
                       whiskerprops=dict(color="0.4", lw=0.7),
                       boxprops=dict(linewidth=0.7),
                       flierprops=dict(marker="o", ms=2.2, mec="none", mfc="0.4"))
    for body, s in zip(parts["boxes"], keep_subjects):
        body.set_facecolor(SUBJECT_COLOR[s]); body.set_edgecolor("0.3"); body.set_alpha(0.7)
    # individual insertion dots overlaid for transparency
    for xi, vals, s in zip(pos, data, keep_subjects):
        jit = np.random.uniform(-0.15, 0.15, len(vals))
        ax.scatter(xi + jit, vals, s=10, color=SUBJECT_COLOR[s],
                   edgecolor="white", lw=0.3, alpha=0.85, zorder=4, rasterized=True)
    ax.set_xticks(pos)
    ax.set_xticklabels([s.replace("sub-", "") for s in keep_subjects],
                       rotation=35, ha="right", fontsize=7.5)
    ax.set_ylabel(label)
    if log_y: ax.set_yscale("symlog", linthresh=1.0)
    ax.spines["left"].set_position(("outward", 4))
    ax.spines["bottom"].set_position(("outward", 4))
fig.suptitle("Figure 5 | Insertion-level quality per subject (each dot = one session × probe)",
             y=0.995, fontweight="bold")
fig.savefig("rfpop_all_fig5_insertion_quality.pdf", dpi=200)
fig.savefig("rfpop_all_fig5_insertion_quality.png")
plt.show(); plt.close(fig); gc.collect()

## Figure 6 — Visual fraction vs depth, one curve per session

Within each session we pool **all probes** (still no letter assumption — depth is a physical quantity, the device label is irrelevant). For each session we compute the fraction of SUA that cleared the visual threshold per 100 µm depth bin. Each curve = one session, coloured by subject. Reveals session-to-session reliability and any depth-aligned drop-off in yield.

In [ ]:
DEPTH_BIN = 100.0
fig, ax = plt.subplots(figsize=(6.5, 4.8))
for asset_id in sorted(UNITS["asset_id"].unique()):
    sub = UNITS[(UNITS["asset_id"] == asset_id) & UNITS["depth_um"].notna()]
    if len(sub) < 30: continue
    s = sub["subject_id"].iloc[0]
    d_lo = float(np.floor(sub["depth_um"].min() / DEPTH_BIN) * DEPTH_BIN)
    d_hi = float(np.ceil(sub["depth_um"].max()  / DEPTH_BIN) * DEPTH_BIN)
    edges = np.arange(d_lo, d_hi + DEPTH_BIN, DEPTH_BIN)
    centers = 0.5 * (edges[:-1] + edges[1:])
    bin_idx = np.clip(np.digitize(sub["depth_um"].values, edges) - 1, 0, len(centers) - 1)
    n_bin = np.bincount(bin_idx, minlength=len(centers)).astype(float)
    k_bin = np.bincount(bin_idx, weights=sub["is_visual"].values.astype(float),
                        minlength=len(centers))
    fr = np.where(n_bin >= 5, k_bin / np.maximum(n_bin, 1), np.nan)
    ax.plot(fr, centers, color=SUBJECT_COLOR[s], lw=0.9, alpha=0.55)
ax.invert_yaxis()
ax.set_xlabel(f"Visually-responsive fraction (peak ΔFR ≥ {VIS_THR:g} spk·s$^{{-1}}$)")
ax.set_ylabel("Depth on probe (µm)")
ax.set_xlim(0, 1)
ax.set_title("Figure 6 | Visual fraction vs depth, one line per session (colour = subject)",
             loc="left", fontweight="bold")
from matplotlib.lines import Line2D
ax.legend(handles=[Line2D([0],[0], color=SUBJECT_COLOR[s], lw=2,
                          label=s.replace("sub-","")) for s in subjects],
          loc="center left", bbox_to_anchor=(1.02, 0.5), fontsize=7.0,
          ncol=1, title="Subject", title_fontsize=8)
fig.savefig("rfpop_all_fig6_visfrac_per_session.pdf", dpi=200)
fig.savefig("rfpop_all_fig6_visfrac_per_session.png")
plt.show(); plt.close(fig); gc.collect()

---
Cached intermediates: `results/rf_cache_allprobes/<asset_id[:8]>.pkl`. Outputs:
* `rfpop_all_fig1_insertion_centroids.{pdf,png}`
* `rfpop_all_fig2_per_subject_spread.{pdf,png}`
* `rfpop_all_fig3_pooled_coverage.{pdf,png}`
* `rfpop_all_fig4_ccf_retinotopy.{pdf,png}`  *(if `estimated_x/y/z` populated)*
* `rfpop_all_fig5_insertion_quality.{pdf,png}`
* `rfpop_all_fig6_visfrac_per_session.{pdf,png}`